工欲善其事，必先利其器。 LLM 的开发跟传统的项目开发区别在于，LLM
很多请求是耗时甚至是耗钱的，基础的如调用 OpenAI API，每次都会消费一定的 token。

另外，我们可能会反复调试一段代码来测试最合适的参数和 prompt，如果我们像传统
nodejs
程序一样每次都从头重新跑一次，既耗时也花费比较多。所以我们需要使用适合机器学习和大模型领域的专用开发工具。

## Deno 与 Jupyter Notebook

Jupyter Notebook
的核心是代码块，每个代码块作为一个整体去执行，并且可以多次反复执行。在代码快的左侧，是执行顺序的标记，指这个代码块被执行的顺序。

有了 Jupyter NoteBook，我们就可以节约 费事/费钱
的请求，并且基于某个运行结果的输出，在后面的代码块中，不断尝试各种解析或者处理方式。同时，也非常方便结合
markdown 来做一些笔记，获得比在注释里记录更方便的学习体验。

### 配置

Jupyter Notebook 项目开始以 python 为主，后续 deno 提供了 js/ts Kernel
的支持，所以我们需要分别安装这两个。这里以 Mac 环境演示，如果是 win/linux
可以参考后附的链接进行安装。

```bash
deno jupyter --install
```

> 参考链接：
>
> - [Deno Jupyter Kernel](https://docs.deno.com/runtime/reference/cli/jupyter/)
> - [Jupyter Notebook 安装](https://jupyter.org/install)

## 相关包安装

- `deno install npm:lodash`
- `deno install npm:@langchain/deepseek`
- `deno install npm:@langchain/core`

In [1]:
import _ from "lodash";

In [2]:
const a = _.random(0, 5);
a;

3

In [1]:
import { ChatDeepSeek } from "@langchain/deepseek";
import { HumanMessage } from "@langchain/core/messages";

const ds = new ChatDeepSeek({
  apiKey: Deno.env.get("DEEPSEEK_API_KEY")!,
  model: "deepseek-chat",
});

await ds.invoke([
  new HumanMessage("你好?"),
]);

AIMessage {
  "id": "08484ee8-92dd-44f2-abb5-a88df4e13f57",
  "content": "你好！很高兴见到你！😊 我是DeepSeek，由深度求索公司创造的AI助手。无论你有什么问题、需要帮助，还是想聊聊天，我都很乐意为你提供帮助！\n\n今天有什么我可以为你做的吗？比如：\n- 回答一些问题\n- 帮你分析或解释某些概念\n- 协助写作或创作\n- 解决学习或工作中的难题\n- 或者就是简单地聊聊天\n\n请随时告诉我你需要什么帮助！✨",
  "additional_kwargs": {},
  "response_metadata": {
    "tokenUsage": {
      "promptTokens": 6,
      "completionTokens": 96,
      "totalTokens": 102
    },
    "finish_reason": "stop",
    "model_provider": "deepseek",
    "model_name": "deepseek-chat",
    "usage": {
      "prompt_tokens": 6,
      "completion_tokens": 96,
      "total_tokens": 102,
      "prompt_tokens_details": {
        "cached_tokens": 0
      },
      "prompt_cache_hit_tokens": 0,
      "prompt_cache_miss_tokens": 6
    },
    "system_fingerprint": "fp_eaab8d114b_prod0820_fp8_kvcache"
  },
  "tool_calls": [],
  "invalid_tool_calls": [],
  "usage_metadata": {
    "output_tokens": 96,
    "input_tokens": 6,
    "total_tokens": 102,
    "input_token_details": {
   

为了方便展示，我们会加入一个简单的 StringOutputParser
来处理输出，你可以简单的理解为将 Deepseek
返回的复杂对象提取出最核心的字符串，更详细的 OutputParser
相关介绍会在后续章节中展开。组成一个最基础的 Chain 来演示， Runnable
中各个调用方式

In [3]:
import { ChatDeepSeek } from "@langchain/deepseek";
import { HumanMessage } from "@langchain/core/messages";
import { StringOutputParser } from "@langchain/core/output_parsers";

const deepseekModel = new ChatDeepSeek({
  apiKey: Deno.env.get("DEEPSEEK_API_KEY")!,
  model: "deepseek-chat",
});
const outputParser = new StringOutputParser();
const ds = deepseekModel.pipe(outputParser);

In [4]:
await ds.invoke([
  new HumanMessage("你好，请用一句话介绍一下你自己。"),
  new HumanMessage("给我讲个笑话。"),
]);

"你好，我是DeepSeek，一个热情洋溢的AI助手，随时准备为你提供帮助！😊\n" +
  "\n" +
  "笑话来了：为什么数学书总是很忧郁？——因为它有太多“问题”要解决了！📚\n" +
  "\n" +
  "有什么其他我可以帮你的吗？"

## batch

然后我们尝试对这个基础的 Chain 进行批量调用，用起来也非常简单

In [6]:
await ds.batch([
  [new HumanMessage("Tell me a joke")],
  [new HumanMessage("你好，你是谁?")],
]);

[
  "Why don’t scientists trust atoms?  \nBecause they make up everything!",
  "你好！我是DeepSeek，由深度求索公司创造的AI助手！😊\n" +
    "\n" +
    "我是一个纯文本模型，专门设计来帮助大家解答问题、提供信息和协助完成各种任务。虽然我不支持多模态识别，但我可以处理你上传的各种文件（图像、txt、pdf、ppt、word、excel等），从中读取文字信息来帮助你。\n" +
    "\n" +
    "我的一些特点：\n" +
    "- 完全免费使用，没有收费计划\n" +
    "- 支持128K上下文长度\n" +
    "- 可以联网搜索（需要你手动开启搜索功能）\n" +
    "- 有官方App可以下载\n" +
    "- 知识截止到2024年7月\n" +
    "\n" +
    "我很乐意成为你的助手，无论是学习、工作还是日常生活中的问题，都可以随时问我！有什么我可以帮你的吗？✨"
]

## 流式传输

因为 LLM
的很多调用都是一段一段的返回的，如果等到完整地内容再返回给用户，就会让用户等待比较久，影响用户的体验。而
LCEL 开箱就支持 steaming，我们依旧使用我们定义的基础 Chain，就可以直接获得
streaming 的能力

In [7]:
const stream = await ds.stream([
  new HumanMessage("Tell me a joke"),
]);

for await (const chunk of stream) {
  console.log(chunk);
}


Why
 don
't
 scientists
 trust
 atoms
?
  

Because
 they
 make
 up
 everything
!




In [9]:
const stream = await ds.streamLog([
  new HumanMessage("Tell me a joke"),
]);

for await (const chunk of stream) {
  console.log(chunk);
}

RunLogPatch {
  ops: [
    {
      op: "replace",
      path: "",
      value: {
        id: "019cb465-c9e4-722d-ba80-6bc8a262a973",
        name: "RunnableSequence",
        type: "chain",
        streamed_output: [],
        final_output: undefined,
        logs: {}
      }
    }
  ]
}
RunLogPatch {
  ops: [
    {
      op: "add",
      path: "/logs/ChatDeepSeek",
      value: {
        id: "019cb465-c9f4-7709-b14d-876719f0aa4c",
        name: "ChatDeepSeek",
        type: "llm",
        tags: [ "seq:step:1" ],
        metadata: {
          ls_provider: "openai",
          ls_model_name: "deepseek-chat",
          ls_model_type: "chat",
          ls_temperature: undefined,
          ls_max_tokens: undefined,
          ls_stop: undefined,
          versions: [Object]
        },
        start_time: "2026-03-03T15:51:35.668Z",
        streamed_output: [],
        streamed_output_str: [],
        final_output: undefined,
        end_time: undefined
      }
    }
  ]
}
RunLogPatch {
  ops